# Candidate cluster profiling

## 1. Objective

Interpret selected K-Means solutions in original customer-behavior units. Clustering uses transformed features; profiling uses the original customer table.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'backend'))
from app.services.cluster_profiling import BEHAVIORAL_FEATURES, profile_candidate

processed_dir = project_root / 'data' / 'processed'
original = pd.read_csv(processed_dir / 'customer_features.csv')
candidates = [('standard', 6), ('standard', 7), ('log_standard', 2),
              ('log_standard', 3), ('log_standard', 4),
              ('log_standard', 5), ('log_robust', 2)]

## 2. Candidate models

All candidates use random_state 42 and n_init 20. CustomerID is excluded from K-Means.

In [ ]:
results = {}
for strategy, k in candidates:
    transformed = pd.read_csv(processed_dir / f'clustering_features_{strategy}.csv')
    results[(strategy, k)] = profile_candidate(transformed, original, strategy, k, 42)
profiles = pd.concat([result.profiles for result in results.values()], ignore_index=True)
print(f'Candidate solutions: {len(results)}')
print(f'Total cluster profiles: {len(profiles)}')

## 3. Cluster-size analysis

In [ ]:
sizes = profiles.assign(configuration=lambda x: x.preprocessing_strategy + '_k' + x.k.astype(str))
fig, axis = plt.subplots(figsize=(12, 6))
labels = sizes['configuration'] + ':c' + sizes['cluster'].astype(str)
axis.bar(labels, sizes['cluster_percentage'], color='#2563eb')
axis.set_ylabel('Customers (%)')
axis.set_title('Cluster sizes across candidate solutions')
axis.tick_params(axis='x', rotation=75)
fig.tight_layout()
plt.show()

## 4. Cluster behavioral profiles

In [ ]:
median_columns = [f'median_{feature}' for feature in BEHAVIORAL_FEATURES]
display(profiles[['preprocessing_strategy', 'k', 'cluster', 'customer_count', 'cluster_percentage', *median_columns]].round(2))
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
profile_labels = profiles.preprocessing_strategy + '_k' + profiles.k.astype(str) + ':c' + profiles.cluster.astype(str)
for feature, axis in zip(BEHAVIORAL_FEATURES, axes.flat):
    axis.plot(range(len(profiles)), profiles[f'median_{feature}'], marker='o', linewidth=1)
    axis.set_title(f'Median {feature}')
    if feature in {'MonetaryValue', 'TotalItems', 'AverageOrderValue', 'AverageItemsPerOrder'}:
        axis.set_yscale('log')
    axis.set_xticks(range(len(profiles)), profile_labels, rotation=90, fontsize=6)
fig.suptitle('Original-unit cluster medians; skewed monetary and volume axes use log scale')
fig.tight_layout()
plt.show()

### Standardized heatmap relative to the overall population

In [ ]:
overall_median = original[list(BEHAVIORAL_FEATURES)].median()
overall_iqr = original[list(BEHAVIORAL_FEATURES)].quantile(.75) - original[list(BEHAVIORAL_FEATURES)].quantile(.25)
median_matrix = profiles[median_columns].copy()
median_matrix.columns = list(BEHAVIORAL_FEATURES)
relative = (median_matrix - overall_median) / overall_iqr
fig, axis = plt.subplots(figsize=(12, 10))
image = axis.imshow(relative.clip(-3, 3), cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
axis.set_xticks(range(len(BEHAVIORAL_FEATURES)), BEHAVIORAL_FEATURES, rotation=45, ha='right')
axis.set_yticks(range(len(profiles)), profile_labels)
fig.colorbar(image, ax=axis, label='Difference from overall median / overall IQR')
axis.set_title('Cluster medians relative to the overall customer population')
fig.tight_layout()
plt.show()

## 5. Micro-cluster audit

In [ ]:
audit_columns = ['cluster', 'CustomerID', 'Country', 'MonetaryValue', 'TotalItems',
                 'Frequency', 'UniqueProducts', 'AverageOrderValue',
                 'AverageItemsPerOrder', 'CustomerLifetimeDays']
for key in [('standard', 6), ('standard', 7)]:
    result = results[key]
    micro_clusters = result.profiles.loc[result.profiles.cluster_percentage < 1, 'cluster']
    micro = result.assignments[result.assignments.cluster.isin(micro_clusters)]
    print(f'{key[0]}, k={key[1]}: {len(micro)} customers in clusters {micro_clusters.tolist()}')
    display(micro[audit_columns].sort_values(['cluster', 'MonetaryValue'], ascending=[True, False]))

## 6. Recency and frequency versus MonetaryValue

In [ ]:
for x_feature in ['Recency', 'Frequency']:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for axis, ((strategy, k), result) in zip(axes.flat, results.items()):
        scatter = axis.scatter(result.assignments[x_feature], result.assignments['MonetaryValue'],
                               c=result.assignments['cluster'], cmap='tab10', s=8, alpha=.45)
        axis.set_title(f'{strategy}, k={k}')
        axis.set_xlabel(x_feature)
        axis.set_ylabel('MonetaryValue')
        axis.set_yscale('log')
    axes.flat[-1].axis('off')
    fig.suptitle(f'{x_feature} vs MonetaryValue in original units; MonetaryValue uses log scale')
    fig.tight_layout()
    plt.show()

## 7. Candidate segment interpretations

Numerical profiles support plain descriptions such as inactive low-activity, moderate developing, recent established repeat, and extreme wholesale-scale customers. Full label rationales are documented in `docs/cluster_profiling.md`. Cluster numbers are arbitrary.

## 8. Comparison of interpretability

Log-standard k=2 is too coarse. K=3 adds a balanced moderate group. K=4 meaningfully separates recent from long-inactive one-date customers. K=5 begins to fragment low-activity behavior. Standard k=6 and k=7 isolate extreme micro-clusters and are less suitable as primary customer typologies.

## 9. Preliminary recommendation

Advance log-standard k=3 and k=4 for later cross-algorithm comparison. This reduces the candidate set using interpretability and balance but does not select a final model.